# Kaggriculture Resumable GPU training

This notebook clones the Kaggriculture training code from GitHub, uses one Colab GPU for compact-policy updates, and uses two CPU workers for isolated simulator rollouts. Checkpoints, trajectories, and exported artifacts are stored on Google Drive so the run can resume after a disconnect.

Staged PPO workflow: run with 16, evaluate development, then change the target to 32 and rerun the training cell. Candidate artifacts are retained separately, so each stage can be evaluated without overwriting the existing current artifact. A development discard means continue training; it is not a promotion. Holdout evaluation is a separate, disjoint step and runs only after a complete development report promotes the candidate.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

repo_url = 'https://github.com/dzlab/kaggle.git'
repo_branch = 'feat/kaggriculture-agent'
clone_root = Path('/content/kaggle')
project_root = clone_root / 'Kaggriculture'
drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    drive.mount('/content/drive')
else:
    print(f'Drive already mounted at {drive_root}')
os.chdir('/content')
if clone_root.exists():
    shutil.rmtree(clone_root)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', repo_branch,
    '--single-branch', repo_url, str(clone_root),
], check=True)
print(f'Cloned {repo_url} ({repo_branch}) into {project_root}')

In [ ]:
%cd /content/kaggle/Kaggriculture
!pip -q install -e '.[training,observability]'
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime in Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/kagriculture-training')
run_dir.mkdir(parents=True, exist_ok=True)
trajectory_path = run_dir / 'bootstrap-trajectories.jsonl'

# CPU-bound simulator collection; the model update below runs on CUDA.
!python scripts/collect_trajectories.py --seeds 8 --start-seed 0 --steps 96 --opponents pass random starter --seats 0 1 --workers 2 --output {trajectory_path}

In [ ]:
import importlib
import scripts.train_policy as train_policy
importlib.reload(train_policy)
import scripts.colab_train as colab_train
importlib.reload(colab_train)
from scripts.export_policy import export_checkpoint
from scripts.train_policy import make_fresh_rollout_fn, train_behavior_clone
from scripts.telemetry import DEFAULT_WANDB_ENTITY, DEFAULT_WANDB_PROJECT, TrainingTelemetry

# Change this one visible target for the next bounded PPO stage.
ppo_target_steps = 16
candidate_tag = f"ppo{ppo_target_steps}"
stage_checkpoint_path = run_dir / f"policy-{candidate_tag}.pt"
stage_artifact_path = run_dir / f"policy-{candidate_tag}.json"
current_checkpoint_path = run_dir / 'policy.pt'

training_steps = 25
training_batch_size = 256
training_seed = 7
training_device = 'cuda'
training_checkpoint_interval = 25
training_prior_checkpoint = None
training_offline_ppo_fallback = False
import wandb
try:
    from google.colab import userdata
    wandb_api_key = os.environ.get('WANDB_API_KEY') or userdata.get('WANDB_API_KEY')
except Exception:
    wandb_api_key = os.environ.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Configure a Colab secret named WANDB_API_KEY before training.')
wandb.login(key=wandb_api_key, relogin=False, quiet=True)
wandb_entity = DEFAULT_WANDB_ENTITY
telemetry_project = DEFAULT_WANDB_PROJECT
wandb_run_name = f'{candidate_tag}-gpu'
training_metrics_path = run_dir / f'{candidate_tag}-training-metrics.jsonl'
training_telemetry = TrainingTelemetry(
    training_metrics_path,
    enable_wandb=True,
    wandb_project=telemetry_project,
    wandb_entity=wandb_entity,
    wandb_run_name=wandb_run_name,
    wandb_config={
        'candidate_tag': candidate_tag,
        'ppo_target_steps': ppo_target_steps,
        'training_steps': training_steps,
        'batch_size': training_batch_size,
        'seed': training_seed,
        'device': training_device,
        'workers': 2,
    },
    strict=True,
)
print('W&B run:', training_telemetry.wandb_url or 'initialized')
def record_training_event(event, metrics=None, **values):
    training_telemetry(
        event, metrics, candidate_tag=candidate_tag,
        ppo_target_steps=ppo_target_steps, **values,
    )
training_contract = train_policy.build_training_contract(
    input_path=trajectory_path,
    steps=training_steps,
    batch_size=training_batch_size,
    seed=training_seed,
    ppo_steps=ppo_target_steps,
    device=training_device,
    checkpoint_interval=training_checkpoint_interval,
    prior_checkpoint=training_prior_checkpoint,
    offline_ppo_fallback=training_offline_ppo_fallback,
)

# Build the later-stage league from only compatible learned checkpoints.
# Malformed, truncated, or version-incompatible Drive files are skipped.
prior_checkpoint_candidates = (
    current_checkpoint_path,
    *sorted(run_dir.glob('policy-ppo*.pt')),
)
compatible_prior_checkpoints = []
for prior_checkpoint in prior_checkpoint_candidates:
    try:
        prior_selection = colab_train.select_resume_checkpoint(
            (prior_checkpoint,), training_contract=training_contract,
            diagnostic=lambda message: print(message),
        )
    except ValueError as exc:
        print(f'Skipping malformed or incompatible checkpoint {prior_checkpoint}: {exc}')
        continue
    if prior_selection is not None:
        compatible_prior_checkpoints.append(prior_selection.path)
training_opponent_pool = train_policy.OpponentPool(
    previous_checkpoints=compatible_prior_checkpoints,
)
print('Compatible learned opponents:', compatible_prior_checkpoints)

resume_selection = colab_train.select_resume_checkpoint(
    (current_checkpoint_path, *sorted(run_dir.glob('policy-ppo*.pt'))),
    training_contract=training_contract,
)
if resume_selection is None:
    saved_ppo_target = None
    resume_checkpoint = None
else:
    saved_ppo_target = resume_selection.ppo_target_steps
    resume_checkpoint = resume_selection.path
allow_ppo_extension = saved_ppo_target is not None and saved_ppo_target < ppo_target_steps
print('Stage:', candidate_tag, 'target PPO steps:', ppo_target_steps)
print('Resume checkpoint:', resume_checkpoint or 'none; starting a new run')
print('allow_ppo_extension:', allow_ppo_extension)

def export_current(*, output_path, **_kwargs):
    export_checkpoint(stage_checkpoint_path, output_path)
    return output_path

fresh_rollout = make_fresh_rollout_fn(
    run_directory=run_dir / 'ppo-rollouts',
    candidate_artifact_callback=export_current,
    seeds=(0, 1, 2, 3),
    steps=97,
    workers=2,
)

metadata = train_behavior_clone(
    input_path=trajectory_path,
    output_path=stage_checkpoint_path,
    steps=training_steps,
    batch_size=training_batch_size,
    seed=training_seed,
    ppo_steps=ppo_target_steps,
    device=training_device,
    checkpoint_interval=training_checkpoint_interval,
    resume_checkpoint=resume_checkpoint,
    allow_ppo_extension=allow_ppo_extension,
    prior_checkpoint=training_prior_checkpoint,
    opponent_pool=training_opponent_pool,
    rollout_fn=fresh_rollout,
    offline_ppo_fallback=training_offline_ppo_fallback,
    candidate_artifact=stage_artifact_path,
    telemetry_callback=record_training_event,
)
export_checkpoint(stage_checkpoint_path, stage_artifact_path)
print(metadata)
print('checkpoint:', stage_checkpoint_path)
print('artifact:', stage_artifact_path)
training_telemetry.finish()

In [ ]:
# Run the complete development gate for this candidate.
# The evaluator writes a report for both valid discards and failures; inspect the report.
import json
import subprocess
import sys

development_seed_values = tuple(range(0, 4))
development_opponents = ('pass', 'random', 'starter')
development_min_valid_games = len(development_seed_values) * len(development_opponents)
development_report_path = run_dir / f'{candidate_tag}-development-evaluation.json'
def evaluation_report_is_complete(report, *, identity, seed_values):
    completeness = report.get('matrix_completeness', {})
    records = report.get('records', {})
    expected_matrix = report.get('expected_matrix', [])
    return (
        report.get('schema_version') == 1
        and report.get('artifact', {}).get('identity') == identity
        and report.get('configuration', {}).get('seed_values') == list(seed_values)
        and set(completeness) == {'current', identity}
        and set(records) == {'current', identity}
        and isinstance(expected_matrix, list)
        and all(
            isinstance(value, dict)
            and value.get('expected_count') == len(expected_matrix)
            and not any(value.get(field) for field in ('missing', 'duplicate', 'extra', 'invalid_records'))
            for value in completeness.values()
        )
        and all(
            isinstance(records[key], list)
            and len(records[key]) == len(expected_matrix)
            for key in ('current', identity)
        )
    )

development_evaluation = subprocess.run(
    [
        sys.executable, 'scripts/evaluate_artifact.py',
        '--artifact', str(stage_artifact_path),
        '--identity', candidate_tag,
        '--seeds', str(len(development_seed_values)),
        '--start-seed', str(development_seed_values[0]),
        '--steps', '96',
        '--opponents', *development_opponents,
        '--seats', '0', '1',
        '--workers', '2',
        '--min-valid-games', str(development_min_valid_games),
        '--output', str(development_report_path),
    ],
    check=False,
)
if not development_report_path.exists():
    raise RuntimeError(
        f'Development evaluator exited {development_evaluation.returncode} without writing {development_report_path}'
    )
development_report = json.loads(development_report_path.read_text(encoding='utf-8'))
development_decision = development_report.get('decision', {})
development_evaluation_complete = (
    evaluation_report_is_complete(
        development_report, identity=candidate_tag, seed_values=development_seed_values,
    )
    and development_decision.get('status') in {'promote', 'discard'}
)
development_evaluation_promoted = (
    development_evaluation_complete
    and development_decision.get('status') == 'promote'
)
print('Development report:', development_report_path)
print('Development status:', development_decision.get('status'))
if development_evaluation_promoted:
    print('Development report is complete and promotes this candidate; holdout may run next.')
else:
    print('Development candidate discarded; discard means continue training, not promotion.')
    print('Continue with another PPO stage; this candidate is not promoted and this report is not promotion evidence.')


In [ ]:
# Run holdout only after the development report is complete and promotes.
# These seeds are intentionally separate from the development matrix.
holdout_seed_values = (100, 101)
holdout_opponents = ('pass', 'random', 'starter')
holdout_min_valid_games = len(holdout_seed_values) * len(holdout_opponents)
assert not set(holdout_seed_values) & set(development_seed_values)
assert holdout_seed_values != development_seed_values
holdout_report_path = run_dir / f'{candidate_tag}-holdout-evaluation.json'
if not development_report_path.exists():
    raise FileNotFoundError(
        f'Run the development evaluation cell first; report missing: {development_report_path}'
    )
development_report = json.loads(development_report_path.read_text(encoding='utf-8'))
development_decision = development_report.get('decision', {})
development_evaluation_promoted = (
    evaluation_report_is_complete(
        development_report, identity=candidate_tag, seed_values=development_seed_values,
    )
    and development_decision.get('status') == 'promote'
)
if development_evaluation_promoted:
    holdout_evaluation = subprocess.run(
        [
            sys.executable, 'scripts/evaluate_artifact.py',
            '--artifact', str(stage_artifact_path),
            '--identity', candidate_tag,
            '--seeds', str(len(holdout_seed_values)),
            '--start-seed', str(holdout_seed_values[0]),
            '--steps', '96',
            '--opponents', *holdout_opponents,
            '--seats', '0', '1',
            '--workers', '2',
            '--min-valid-games', str(holdout_min_valid_games),
            '--output', str(holdout_report_path),
        ],
        check=False,
    )
    if not holdout_report_path.exists():
        raise RuntimeError(
            f'Holdout evaluator exited {holdout_evaluation.returncode} without writing {holdout_report_path}'
        )
    holdout_report = json.loads(holdout_report_path.read_text(encoding='utf-8'))
    holdout_decision = holdout_report.get('decision', {})
    holdout_evaluation_complete = (
        evaluation_report_is_complete(
            holdout_report, identity=candidate_tag, seed_values=holdout_seed_values,
        )
        and holdout_decision.get('status') in {'promote', 'discard'}
    )
    if not holdout_evaluation_complete:
        raise RuntimeError('Holdout report is incomplete or has no valid decision status')
    print('Holdout report:', holdout_report_path)
    print('Holdout status:', holdout_decision.get('status'))
else:
    print('Holdout evaluation skipped: development report is not complete/promote.')


In [ ]:
# Verify the exported dependency-free candidate in the local simulator.
import re
import subprocess
import sys

smoke = subprocess.run(
    [
        'python', 'scripts/run_local.py',
        '--opponent', 'pass',
        '--seed', '0',
        '--steps', '96',
        '--replay', str(run_dir / 'candidate-smoke.json'),
        '--candidate-artifact', str(stage_artifact_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(smoke.stdout, end='')
# kaggle-environments 1.32.7 probes an unavailable optional OpenSpiel game
# during startup. Remove only that known diagnostic; preserve all other stderr.
clean_stderr = re.sub(
    r"OpenSpiel exception: Unknown game 'python_ant_foraging'\. Available games are:\n.*?\nzerosum\n?",
    '',
    smoke.stderr,
    flags=re.DOTALL,
)
if clean_stderr:
    print(clean_stderr, file=sys.stderr, end='')
print('Smoke test completed successfully. Re-run the training cell after a disconnect; it will resume from the latest compatible PPO stage checkpoint when present.')

In [ ]:
# Plot the local training history; an unavailable or empty log is valid.
import matplotlib.pyplot as plt
from scripts.telemetry import load_metrics

training_events = load_metrics(training_metrics_path)
if not training_events:
    print(f'No {candidate_tag} training telemetry found at {training_metrics_path}')
else:
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=False)
    fig.suptitle(f'{candidate_tag} training telemetry')
    for axis, event_name, metric_name, title in (
        (axes[0], 'behavior_clone', 'loss', 'Behavior-cloning loss'),
        (axes[1], 'ppo', 'policy_loss', 'PPO policy loss'),
    ):
        rows = [
            row for row in training_events
            if row.get('event') == event_name and metric_name in row
        ]
        if rows:
            axis.plot([row.get('step', index) for index, row in enumerate(rows)], [row[metric_name] for row in rows], marker='o')
            axis.set_title(f'{candidate_tag} — {title}')
            axis.set_xlabel('Step')
            axis.set_ylabel(metric_name)
            axis.grid(True)
        else:
            axis.text(0.5, 0.5, f'No {event_name} {metric_name} data', ha='center', va='center')
            axis.set_axis_off()
    fig.tight_layout()
    plt.show()
